In [24]:
import pandas as pd
import numpy as np
import imageio
import h5py
import os, json, sys
import math, random
import shutil

## functions

### umap functions

In [25]:
def scale_to_int_coords (array,  target_min, target_max):
    scaled_array = ((array - array.min()) / (array.max() - array.min())) * (target_max - target_min) + target_min
    int_coords = np.round(scaled_array).astype(int)
    return int_coords

In [26]:
# return a numpy array with columns as "UMAP1_scaleInt", "UMAP2_scaleInt", "tile_x", "tile_y"
def convert_UMAP (arr1_size, UMAP):
    # UMAP is 2D numpy array
    UMAP1_min = np.min(UMAP[:, 0])
    UMAP1_max = np.max(UMAP[:, 0])
    UMAP2_min = np.min(UMAP[:, 1])
    UMAP2_max = np.max(UMAP[:, 1])
    UMAP1_size = UMAP1_max - UMAP1_min
    UMAP2_size = UMAP2_max - UMAP2_min
    arr2_size = int(UMAP2_size/UMAP1_size * arr1_size)

    UMAP_df =pd.DataFrame(np.zeros((UMAP.shape[0], 4)), columns=["UMAP1_scaleInt", "UMAP2_scaleInt", "tile_x", "tile_y"])
    
    UMAP_df["UMAP1_scaleInt"] = scale_to_int_coords (UMAP[:, 0], 0, arr1_size)
    UMAP_df["UMAP2_scaleInt"] = scale_to_int_coords (UMAP[:, 1], 0, arr2_size)
    
    # Compute tile coordinates
    UMAP_df["tile_x"] = UMAP_df["UMAP1_scaleInt"] // tilesize
    UMAP_df["tile_y"] = UMAP_df["UMAP2_scaleInt"] // tilesize
    
    # number of non-empty pixels
    pixel_counts = UMAP_df.groupby(['UMAP1_scaleInt', 'UMAP2_scaleInt']).size()
    
    # number of non-empty tiles
    tile_counts = UMAP_df.groupby(['tile_x', 'tile_y']).size()

    # total number of tiles
    total_tile = np.max(UMAP_df['tile_x']) * np.max(UMAP_df['tile_y'])
    
    # Compute average for non-empty tiles
    avg_points_per_non_empty_tile = tile_counts.mean()

    UMAP_df = UMAP_df.to_numpy()
    
    return UMAP_df

In [27]:
# return umap data loss rate if with arr1_size
def UMAP_loss_rate (arr1_size, UMAP):
    # UMAP is 2D numpy array
    UMAP1_min = np.min(UMAP[:, 0])
    UMAP1_max = np.max(UMAP[:, 0])
    UMAP2_min = np.min(UMAP[:, 1])
    UMAP2_max = np.max(UMAP[:, 1])
    UMAP1_size = UMAP1_max - UMAP1_min
    UMAP2_size = UMAP2_max - UMAP2_min
    arr2_size = int(UMAP2_size/UMAP1_size * arr1_size)

    UMAP_df =pd.DataFrame(np.zeros((UMAP.shape[0], 4)), columns=["UMAP1_scaleInt", "UMAP2_scaleInt", "tile_x", "tile_y"])
    
    UMAP_df["UMAP1_scaleInt"] = scale_to_int_coords (UMAP[:, 0], 0, arr1_size)
    UMAP_df["UMAP2_scaleInt"] = scale_to_int_coords (UMAP[:, 1], 0, arr2_size)
    
    # Compute tile coordinates
    UMAP_df["tile_x"] = UMAP_df["UMAP1_scaleInt"] // tilesize
    UMAP_df["tile_y"] = UMAP_df["UMAP2_scaleInt"] // tilesize
    
    # number of non-empty pixels
    pixel_counts = UMAP_df.groupby(['UMAP1_scaleInt', 'UMAP2_scaleInt']).size()
    
    # number of non-empty tiles
    tile_counts = UMAP_df.groupby(['tile_x', 'tile_y']).size()

    # total number of tiles
    total_tile = np.max(UMAP_df['tile_x']) * np.max(UMAP_df['tile_y'])
    
    # Compute average for non-empty tiles
    avg_points_per_non_empty_tile = tile_counts.mean()

    loss_rate = 1-len(pixel_counts)/UMAP_df.shape[0]
    print(arr1_size, arr2_size, len(pixel_counts), len(tile_counts), total_tile, avg_points_per_non_empty_tile, loss_rate)
    
    return loss_rate

### code dictionary.  save value =0 as background function
return categories_int, cat_list

In [28]:
# data : metadata df
# feature: metadata df column
def findCodes(data, feature): 
    categories = data[feature].to_numpy()  # Replace with actual column name
    print (feature)
    print(categories)
    
    from sklearn.preprocessing import LabelEncoder
    
    # Encode categories as integers starting from 1
    le = LabelEncoder()
    mask = ~pd.isna(categories)
    categories_int = np.zeros_like(categories, dtype=int)
    categories_int[mask] = le.fit_transform(categories[mask]) + 1
    print(len(le.classes_))
    
    int_to_cat_dict = {i + 1: str(cat) for i, cat in enumerate(le.classes_)}
    cat_list = list(int_to_cat_dict.values())
    cat_list.insert(0, "background")
    
    print(json.dumps(cat_list, indent=2))
    print(len(cat_list))

    return categories_int, cat_list

In [29]:
def binContinuous(data, feature, n_bins=10):
    values = data[feature].to_numpy(dtype=float)
    categories_int = np.zeros(len(values), dtype=int)

    mask = ~np.isnan(values)
    valid_values = values[mask]

    vmin, vmax = valid_values.min(), valid_values.max()
    step_raw = (vmax - vmin) / n_bins

    # Round step up to a nice number so bin edges land on round values
    magnitude = 10 ** math.floor(math.log10(step_raw))
    nice_step = next(m * magnitude for m in [1, 2, 2.5, 5, 10] if m * magnitude >= step_raw)
    nice_min = math.floor(vmin / nice_step) * nice_step
    nice_max = math.ceil(vmax / nice_step) * nice_step
    actual_n_bins = round((nice_max - nice_min) / nice_step)
    bins = np.array([nice_min + i * nice_step for i in range(actual_n_bins + 1)])

    bin_indices = np.digitize(valid_values, bins)
    bin_indices = np.clip(bin_indices, 1, actual_n_bins)

    # Remove empty bins so the legend never has zero-count entries
    bin_counts = np.bincount(bin_indices, minlength=actual_n_bins + 1)[1:]
    non_empty = np.where(bin_counts > 0)[0]

    # Derive decimal places from nice_step so all labels use consistent formatting
    if nice_step >= 1:
        fmt = ".0f"
    else:
        n_dec = max(1, -int(math.floor(math.log10(nice_step))))
        fmt = f".{n_dec}f"

    all_labels = [f"{bins[i]:{fmt}}–{bins[i+1]:{fmt}}" for i in range(actual_n_bins)]
    kept_labels = [all_labels[i] for i in non_empty]

    old_to_new = {old + 1: new + 1 for new, old in enumerate(non_empty)}
    bin_indices_remapped = np.vectorize(old_to_new.get)(bin_indices)
    categories_int[mask] = bin_indices_remapped

    cat_list = ["background"] + kept_labels

    print(f"{feature}: {len(kept_labels)} bins (of {actual_n_bins}), {mask.sum()} valid / {len(values)} total")
    return categories_int, cat_list

### depth_bit function

In [30]:
# depth_bit
# 16, #65536.0
# 8, #256
def find_depth_bit (cat_list):
    if len(cat_list) <= 256:
        depth_bit =8
    else:
        depth_bit = 16
    return depth_bit

### build base layer function
array_2d_cat:  xi, yi, tilex, tiley, pixel_totalcount, pixel_cats

In [31]:
def determine_maxLevel (h, w):
    max_s = max(h, w)
    max_level = math.ceil(math.log2(max_s)) - math.log2(tilesize)
    return int(max_level)

In [32]:
def build_array_2d_cat (UMAP_df, tilesize):
    #from collections import defaultdict
    #from scipy import sparse

    x = UMAP_df[:,0]
    y = UMAP_df[:,1]
    
    # Determine height and width as the next multiple of tilesize
    height = ((y.max() + tilesize) // tilesize) * tilesize
    width = ((x.max() + tilesize) // tilesize) * tilesize
    print(f"Final shape: h {height} x w {width}")
    
    max_level = determine_maxLevel(height, width)   # not the number of levels (e.g. 8), but the max level number (8-1=7)
    print ("Levels:", max_level +1, "Max Level:", max_level)
    
    UMAP_cat = np.hstack([UMAP_df, categories_int.reshape(-1, 1)]) #UMAP1_int, UMAP2_int, tile_x, tile_y, category_int
    UMAP_cat.shape

    # build UMAP_cat
    # build a pixel_keys, and pixel_counts
    # Step 1: Take first two columns
    keys = UMAP_cat[:, :2]
    
    # Step 2: Use np.unique with return_counts
    unique_keys_first, pixel_totalcount = np.unique(keys, axis=0, return_counts=True)
    unique_keys_first, pixel_totalcount, len(pixel_totalcount)

    # build a pixel_keys, and pixel_cats
    keys = UMAP_cat[:, :2]
    cats = UMAP_cat[:, 4]
    
    # Step 1: lexsort keys to group
    sorted_idx = np.lexsort((keys[:, 0], keys[:, 1]))
    keys_sorted = keys[sorted_idx]
    cats_sorted = cats[sorted_idx]
    
    # Step 2: find unique keys and where they start
    unique_keys_second, start_idx, counts = np.unique(keys_sorted, axis=0, return_index=True, return_counts=True)
    
    assert np.array_equal(unique_keys_first, unique_keys_second)
    
    # Step 3: pick a random row from each group
    np.random.seed(42)  # optional for reproducibility
    random_offsets = np.random.randint(0, counts)
    random_idx = start_idx + random_offsets
    pixel_cats = cats_sorted[random_idx]
    unique_keys_second, pixel_cats, len(pixel_cats)

    # put pixel_x, pixel_y, tile_x, tile_y, pixel_counts, pixel_cats together into a 6 column numpy array
    array_2d_cat = np.concatenate(
        [unique_keys_first, unique_keys_first//tilesize, pixel_totalcount[:, None], pixel_cats[:, None]], axis=1
    )

    return array_2d_cat, max_level

In [33]:
def get_umap_df_dim (UMAP_df):
    x = UMAP_df[:,0]
    y = UMAP_df[:,1]
    
    # Determine height and width as the next multiple of tilesize
    height = ((y.max() + tilesize) // tilesize) * tilesize
    width = ((x.max() + tilesize) // tilesize) * tilesize

    return height, width

### downsample category data function

To downsample a 2D categorical array (where each pixel holds an integer representing a category) by a factor of 2, and assign each block’s value by randomly sampling a non-zero category from the block according to frequency,

In [34]:
def downsample_categorical_by_frequency(arr, pixel_area_equivalent, categories_size, 
                                        density_threshold, block_size=2, background_value=0):
    
    # arr:  pixel_x, pixel_y, tile_x, tile_y, pixel_counts, pixel_cats

    new_xy = arr[:, :2] // block_size
    counts_arr = arr[:, 4]
    cats_arr = arr[:, 5]
    
    unique_keys_first, inverse_idx = np.unique(new_xy, axis=0, return_inverse=True)
    pixel_totalcount = np.bincount(inverse_idx, weights=counts_arr).astype(int)

    # Step 1: lexsort keys to group
    sorted_idx = np.lexsort((new_xy[:, 0], new_xy[:, 1]))
    keys_sorted = new_xy[sorted_idx]
    cats_sorted = cats_arr[sorted_idx]
    
    # Step 2: find unique keys and where they start
    unique_keys_second, start_idx, counts = np.unique(keys_sorted, axis=0, return_index=True, return_counts=True)

    assert np.array_equal(unique_keys_first, unique_keys_second)
    
    # Step 3: pick a random row from each group
    random_offsets = np.random.randint(0, counts)
    random_idx = start_idx + random_offsets
    pixel_cats = cats_sorted[random_idx]

    # Step 4, low density pixel with 50% removal rate
    mask_count = pixel_totalcount/pixel_area_equivalent < density_threshold
    # random 0/1 (True ~50% chance) of same shape
    rand_mask = np.random.rand(len(pixel_totalcount)) < 0.5  
    final_mask = mask_count & rand_mask
    pixel_cats[final_mask] = 0
    
    # Combine into final array
    downsampled = np.concatenate([unique_keys_first, unique_keys_first//tilesize, pixel_totalcount[:, None], pixel_cats[:, None]], axis=1)
    
    return downsampled

### save tile function

In [35]:
from concurrent.futures import ThreadPoolExecutor

def saveAsTiles(array_2d_cat, level,
                tile_size, bitdepth, start_chn, build_dir, target_dir):
    
    # array_2d_cat:  pixel_x, pixel_y, tile_x, tile_y, pixel_counts, pixel_cats
    
    print("building tiles for level", level)

    tiles_keys, inverse_idx = np.unique(
        array_2d_cat[:,2:4], axis=0, return_inverse=True
    )

    def process_tile(k_tile):
        k, (tx, ty) = k_tile
        
        # Get all points in this tile
        rows = np.where(inverse_idx == k)[0]
        
        # Compute local coordinates inside tile
        local_x = array_2d_cat[rows, 0] - tx * tile_size
        local_y = array_2d_cat[rows, 1] - ty * tile_size
        cats = array_2d_cat[rows, 5]
    
        if bitdepth == 16:
            tile = np.zeros((tile_size, tile_size), dtype=np.uint16)
        else:
            tile = np.zeros((tile_size, tile_size), dtype=np.uint8)
    
        tile[local_y, local_x] = cats
    
        rawPNGFile = f"{build_dir}/tile_{level}_{tx*tile_size}_{ty*tile_size}.png"
        imageio.imwrite(rawPNGFile, tile)
    
        dim = f'{tile_size}x{tile_size}'
        finalPNGFile = f'{target_dir}/p{start_chn}-{level}-{ty}-{tx}.png'
        os.system(f'convert {rawPNGFile} -crop {dim} -define png:bit-depth={bitdepth} '
                  f'-depth {bitdepth} -background none -extent {dim} -alpha off {finalPNGFile}')

    # --- Parallel execution ---
    num_cpus = os.cpu_count()
    with ThreadPoolExecutor(max_workers=num_cpus) as executor:  # adjust threads as needed
        executor.map(process_tile, enumerate(tiles_keys))
        

### build metadata function

In [36]:
def write_json(reference_name, viz_label, start_chn, cat_list, feature_type, tilesize, max_level, offset_x, offset_y, image_scalef):
    viz_type = "ordinal" if feature_type == "continuous" else "category"
    # read existing one in
    path = os.path.join(target_dir, "metadata.json")
    if os.path.exists(path):
        with open(path, "r") as f:
            J = json.load(f)
            assert(J["tileSize"] == tilesize)
            assert(J["levels"] == max_level + 1)
            assert(J["offset"] == [float(offset_x), float(offset_y)])
            assert(J["image_scalef"] == float(image_scalef))
            
            for i in range(len(J["phenotypes"]), start_chn + 1):
                J["phenotypes"].append({})  # placeholder
    
            J["phenotypes"][start_chn] = {
                "name": viz_label,
                "type": viz_type,
                "int_to_category": cat_list
            }
    else:
        height, width = get_umap_df_dim (UMAP_df)
        J={}
        J["reference_name"]= reference_name
        J["count"] = UMAP.shape[0]
        
        J["phenotypes"] = [] 
        while len(J["phenotypes"]) <= start_chn:
            J["phenotypes"].append({})  # placeholder
    
        J["phenotypes"][start_chn] = {
            "name": viz_label,
            "type": viz_type,
            "int_to_category": cat_list
        }
        J["size"] = [
            int(width/math.pow(2, max_level)),
            int(height/math.pow(2, max_level))
        ]
        J["tileSize"] = tilesize
        J["levels"] = max_level + 1
        J["fileformat"] = "png"
        J["offset"]= [float(offset_x), float(offset_y) ]
        J["image_scalef"] = float(image_scalef)
    print(J)
    with open(os.path.join(target_dir, "metadata.json"), "w") as f:
        json.dump(J, f, indent=4)

## set up section

In [37]:
# only parameter to set on the fly, the rest is configured in pyramid_config.json
# all input files are expected to be in results_dir
results_dir = "/mnt/efsUCE/HNOCA_Extended_ff9982c2-06cc-4376-930b-186fcbc1fd82/"
pyramid_file = "pyramid_e30_config.json"

In [38]:
# Path to your config file
config_path = os.path.join(results_dir, pyramid_file)

# Open and load the JSON
with open(config_path, "r") as f:
    config = json.load(f)

reference_name = config["reference_name"]
target_dir = config["target_dir"]
umap_file = config["umap_file"]
metadata_file = config["metadata_file"] # expect first column is index

features = config["features"]
feature_labels = config["feature_labels"]
start_chns = config.get("start_chns", list(range(len(features))))
feature_types = config.get("feature_types", ["category"] * len(features))
n_bins_config = config.get("n_bins", 10)
n_bins_list = [n_bins_config] * len(features) if isinstance(n_bins_config, int) else n_bins_config

if not isinstance(features, list):
    raise TypeError("'features' must be a list")
if not isinstance(feature_labels, list):
    raise TypeError("'feature_labels' must be a list")
if not isinstance(start_chns, list):
    raise TypeError("'start_chns' must be a list of indices")
if not isinstance(feature_types, list):
    raise TypeError("'feature_types' must be a list")
assert(len(features) == len(feature_labels))
assert(len(features) == len(feature_types))
assert(len(features) == len(n_bins_list))
assert(all(i < len(features) for i in start_chns))

features = [features[i] for i in start_chns]
feature_labels = [feature_labels[i] for i in start_chns]
feature_types = [feature_types[i] for i in start_chns]
n_bins_list = [n_bins_list[i] for i in start_chns]

build_dir = 'build'
tilesize = 1024

if os.path.exists(build_dir):
    shutil.rmtree(build_dir, ignore_errors=True)
if not os.path.exists(build_dir):
    os.mkdir(build_dir)

os.makedirs(target_dir, exist_ok=True)

In [39]:
start_chns, features, feature_labels

([0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33],
 ['harmonized_cell_label',
  'assay',
  'collection_doi_label',
  'development_stage',
  'disease',
  'publication',
  'suspension_type',
  'tissue',
  'tissue_type',
  'Hallmark_Glycolysis',
  'annot_level_1',
  'annot_level_2',
  'annot_level_2_extended',
  'annot_level_3_rev2',
  'annot_level_4_rev2',
  'annot_region_rev2',
  'assay_differentiation',
  'assay_type_differentiation',
  'cell_line',
  'cell_type',
  'cell_type_original',
  'doi',
  'donor_id',
  'gm',
  'harmonized_cell_label_ontology_term_name',
  'harmonized_cell_label_qc',
  'hnoca_core',
  'id',
  'individual',
  'organoid_age_days',
  'self_reported_ethnicity',
  'sex',
  'state_exact',
  'treatment'],
 ['harmonized_cell_label',
  'assay',
  'collection_doi_label',
  'development_stage',
  'disease',
  'publication',
  'su

## Run

### load metadata and umap files

In [40]:
data = pd.read_csv(os.path.join(results_dir, metadata_file), sep ='\t')

/tmp/ipykernel_704899/1210958689.py:1: DtypeWarning: Columns (1,2,3,4,6,7,8,9,11,17,18,19,20,21,22) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(os.path.join(results_dir, metadata_file), sep ='\t')


In [41]:
# load umap data 
UMAP = np.load(os.path.join(results_dir, umap_file))
UMAP.shape

(1920454, 2)

### determine tile dimension

In [42]:
print("arr1_size", "arr2_size", "pixel_counts", "tile_counts", "total_tile", "avg_points_per_non_empty_tile", "loss_rate")
for arr1_size in [2**i for i in range(13, 20)]: #20
    loss_rate = UMAP_loss_rate (arr1_size, UMAP)
    if loss_rate < 0.03:
        break
print(arr1_size)

arr1_size arr2_size pixel_counts tile_counts total_tile avg_points_per_non_empty_tile loss_rate
8192 6486 1577876 35 48 54870.114285714284 0.17838386131612627
16384 12972 1812868 110 192 17458.672727272726 0.05602112833736195
32768 25944 1890305 380 800 5053.8263157894735 0.015698892032821377
32768


### build base UMAP

In [43]:
UMAP_df = convert_UMAP (arr1_size, UMAP)

### determine scale and offset

In [44]:
array = UMAP[:,0]
target_max = arr1_size
target_min = 0
image_scalef = 1/(array.max() - array.min()) * (target_max - target_min)
offset_x = - array.min()/ (array.max() - array.min()) * (target_max - target_min) + target_min
print ("image_scalef:", image_scalef)
print ("offset_x:", offset_x)

array = UMAP[:,1]
target_max = np.max(UMAP_df[:,1])
offset_y = - array.min()/ (array.max() - array.min()) * (target_max - target_min) + target_min
print("offset_y", offset_y)

image_scalef: 836.1674
offset_x: 17827.76
offset_y 12903.527711391449


## Build tiles

In [45]:
UMAP_df

array([[15213, 13275,    14,    12],
       [15626, 13523,    15,    13],
       [17122, 12301,    16,    12],
       ...,
       [30609, 17621,    29,    17],
       [26216, 25566,    25,    24],
       [26362, 25457,    25,    24]], shape=(1920454, 4))

In [46]:
for feature, start_chn, viz_label, feature_type, n_bins in zip(features, start_chns, feature_labels, feature_types, n_bins_list):
    # code
    if feature_type == "continuous":
        categories_int, cat_list = binContinuous(data, feature, n_bins)
    else:
        categories_int, cat_list = findCodes(data, feature)
    # depth
    depth_bit = find_depth_bit (cat_list)
    # categorical base layer 
    array_2d_cat, max_level = build_array_2d_cat(UMAP_df, tilesize)

    # tiling
    current = array_2d_cat
    if max_level == 4:
        init_density_threshold = 1.0/(2.0) # 5 levels
    if max_level == 5:
        init_density_threshold = 1.0/(4.0) # 6 levels
    if max_level == 6:
        init_density_threshold = 1.0/(4.0) # 7 levels
    elif max_level == 7:
        init_density_threshold = 1.0/(4.0 * 2.0) # 8 levels
    elif max_level == 8:
        init_density_threshold = 1.0/(4.0 * 4.0) # 9 levels
    elif max_level == 9:
        init_density_threshold = 1.0/(4.0 * 4.0 * 4.0) # 10 levels
    elif max_level == 10:
        init_density_threshold = 1.0/(4.0 * 4.0 * 4.0 * 2.0) # 11 levels
    else:
        print ("for max level", max_level +1, "need to figure out the initial density threshold")
    level = max_level
    
    current_base_area = 1
    block_size = 2
    step = 2.0
    threshold_level = max_level
    
    next_base_area =  current_base_area * block_size * block_size
    next_threshold = init_density_threshold/step
    
    print ("Begin downsampling ...")
    categories_size = len(cat_list) - 1   
    while level > 0:
        result =  downsample_categorical_by_frequency( current, next_base_area, categories_size, 
                                                       next_threshold, block_size=block_size)
        level = level - 1 
        print("Downsampled to level:", level, "count", result.shape)

        if level < threshold_level:
            saveAsTiles(result, level, tilesize, depth_bit, start_chn, build_dir, target_dir)
            print("Saved tiles to disk, level", level)
        
        current = result
        next_base_area = next_base_area * block_size * block_size
        next_threshold = next_threshold/step
    
    ## json
    write_json(reference_name, viz_label, start_chn, cat_list, feature_type, tilesize, max_level, offset_x, offset_y, image_scalef)
    
    ## build base layer tiles
    if threshold_level == max_level:
        saveAsTiles(array_2d_cat, max_level, tilesize, depth_bit, start_chn, build_dir, target_dir)

harmonized_cell_label
['Medulla Neural Progenitor Cell' 'Medulla Neural Progenitor Cell'
 'Medulla Neuron NT-GABA' ... 'Dorsal Telencephalic Neuron' 'Glioblast'
 'Dorsal Telencephalic Neural Progenitor Cell']
58
[
  "background",
  "Astrocyte",
  "Cerebellar Neural Progenitor Cell",
  "Cerebellar Neuron NT-GABA",
  "Cerebellar Neuron NT-GLY",
  "Cerebellar Neuron NT-VGLUT",
  "Choroid Plexus",
  "Dorsal Midbrain Neural Progenitor Cell",
  "Dorsal Midbrain Neuron NT-GABA",
  "Dorsal Midbrain Neuron NT-VGLUT",
  "Dorsal Telencephalic Intermediate Progenitor",
  "Dorsal Telencephalic Neural Progenitor Cell",
  "Dorsal Telencephalic Neuron",
  "Dorsal Telencephalic Neuron NT-GABA",
  "Dorsal Telencephalic Neuron NT-VGLUT",
  "Dorsal Telencephalic Neuron NT-VGLUT CFuPN",
  "Dorsal Telencephalic Neuron NT-VGLUT CPN",
  "Dorsal Telencephalic Neuron NT-VGLUT Cajal-Retzius",
  "Dorsal Telencephalic Neuron NT-VGLUT L6b",
  "Dorsal Telencephalic Neuron NT-VGLUT Newborn PN",
  "Dorsal Telencephali